## **TPKAGGLE**



In [74]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier # Un modelo simple para empezar
from sklearn.metrics import roc_auc_score

# Cargar los datos
df_solicitudes_test = pd.read_csv('/content/solicitudes_test.csv')
df_comportamiento = pd.read_csv('/content/comportamiento_bancario.csv')
df_solicitudes = pd.read_csv('/content/solicitudes_train.csv')
df_solicitudes.head()

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,Estado_Final
0,1,12698,7900,0.2333,4461,Consumo,2025-04,1
1,2,20744,17400,0.2507,3889,Vivienda,2025-12,1
2,3,6830,8600,0.2368,4035,Vehiculo,2025-06,0
3,4,5722,13300,0.2826,4018,Vehiculo,2025-07,1
4,5,24719,6600,0.2314,3357,Otros,2025-01,1


In [75]:

df_comportamiento['Mes_Referencia'] = pd.to_datetime(df_comportamiento['Mes_Referencia'])
df_solicitudes['Fecha_Solicitud'] = pd.to_datetime(df_solicitudes['Fecha_Solicitud'])
df_solicitudes_test['Fecha_Solicitud'] = pd.to_datetime(df_solicitudes_test['Fecha_Solicitud'])

def agregar_comportamiento(df_solicitudes_base, df_comportamiento):
    df_temp = pd.merge(
        left=df_solicitudes_base[['ID_Solicitud','ID_Usuario','Fecha_Solicitud']],
        right=df_comportamiento[['ID_Usuario','Mes_Referencia','Saldo_Promedio_Mensual','Dias_Atraso_Otras_Deudas']],
        on='ID_Usuario',
        how='left'
    )
    # Solo meses anteriores a CADA fecha de solicitud puntual
    df_temp = df_temp[df_temp['Mes_Referencia'] < df_temp['Fecha_Solicitud']]
    return df_temp.groupby('ID_Solicitud')[['Saldo_Promedio_Mensual','Dias_Atraso_Otras_Deudas']].mean()

df_comportamiento_agrupado_train = agregar_comportamiento(df_solicitudes, df_comportamiento)
df_cruzado = pd.merge(df_solicitudes, df_comportamiento_agrupado_train, on='ID_Solicitud', how='left')
df_cruzado.sort_values(by='ID_Usuario', ascending=True)

# Para test, con sus PROPIAS fechas (2026-01):
df_comportamiento_agrupado_test = agregar_comportamiento(df_solicitudes_test, df_comportamiento)
df_solicitudes_test_cruzado = pd.merge(
    left=df_solicitudes_test,
    right=df_comportamiento_agrupado_test,
    on='ID_Solicitud',
    how='left'
  )

df_solicitudes_test_cruzado.isnull().sum(),df_cruzado.isnull().sum()

(ID_Solicitud                0
 ID_Usuario                  0
 Monto_Solicitado            0
 Tasa_Interes                0
 Ingreso_Mensual             0
 Motivo_Prestamo             0
 Fecha_Solicitud             0
 Saldo_Promedio_Mensual      0
 Dias_Atraso_Otras_Deudas    0
 dtype: int64,
 ID_Solicitud                0
 ID_Usuario                  0
 Monto_Solicitado            0
 Tasa_Interes                0
 Ingreso_Mensual             0
 Motivo_Prestamo             0
 Fecha_Solicitud             0
 Estado_Final                0
 Saldo_Promedio_Mensual      0
 Dias_Atraso_Otras_Deudas    0
 dtype: int64)

In [76]:
from sklearn.pipeline import Pipeline
from sklearn_pandas import DataFrameMapper
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn import metrics

#Creamos el mapper
mapper = DataFrameMapper([
(['Monto_Solicitado'],[ StandardScaler()]),
(['Saldo_Promedio_Mensual'],[StandardScaler(),IterativeImputer()]),
(['Dias_Atraso_Otras_Deudas'],[StandardScaler(),IterativeImputer()])
])
#Creamos el pipeline
pipeline = Pipeline(
    [('mapper', mapper),
     ('classifier', LogisticRegression(random_state=70))
])


#Entrenamos con train
pipeline.fit(df_cruzado, df_cruzado['Estado_Final'])

/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)


Pipeline(steps=[('mapper',
                 DataFrameMapper(drop_cols=[],
                                 features=[(['Monto_Solicitado'],
                                            [StandardScaler()]),
                                           (['Saldo_Promedio_Mensual'],
                                            [StandardScaler(),
                                             IterativeImputer()]),
                                           (['Dias_Atraso_Otras_Deudas'],
                                            [StandardScaler(),
                                             IterativeImputer()])])),
                ('classifier', LogisticRegression(random_state=70))])

In [77]:
#METRICAS PARA VER PESOS DE VARIABLES
clf = pipeline.named_steps['classifier']
clf.coef_, pipeline.named_steps['mapper'].transformed_names_


(array([[-0.55411505,  0.93489442, -0.86882441]]),
 ['Monto_Solicitado', 'Saldo_Promedio_Mensual', 'Dias_Atraso_Otras_Deudas'])

In [78]:
new_predictions_prob = pipeline.predict_proba(df_solicitudes_test_cruzado)[:, 1]
# AGREGAMOS A LAS SOLICITUDES LA COLUMNA DE PROBABILIDAD DE QUE SEA PAGADO EL PRESTAMO
df_solicitudes_test_cruzado['Prob_Pago'] = new_predictions_prob
df_solicitudes_test_cruzado.sample(5)

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas,Prob_Pago
1746,34163,27530,500,0.2530,1345,Consumo,2026-01-01,2761.266818,3.545455,0.953922
2860,35277,27423,2700,0.2784,4486,Educacion,2026-01-01,9170.629091,8.000000,0.974440
396,32813,28068,5200,0.2864,7637,Salud,2026-01-01,3826.721304,22.130435,0.658262
2608,35025,26625,900,0.2926,1628,Vehiculo,2026-01-01,750.245417,10.291667,0.851501
1524,33941,27722,2300,0.2541,4850,Vivienda,2026-01-01,3636.975909,16.590909,0.810421


In [79]:
#MECANISMO PARA NO PRESTAR DE MáS
monto_solicitado_total = 0
monto_a_ganar=0
#AñADIMOS UNA COLUMNA A LAS SOLICITUDES CON ESTADO = 0
df_solicitudes_test_cruzado['Estado'] = 0
#CREAMOS LISTA DE INDICES DE LAS FILAS DE LAS SOLICITUDES QUE APROBAREMOS
approved_indices = []

#ITERAMOS CADA FILA
for index, row in df_solicitudes_test_cruzado.sort_values(by='Prob_Pago', ascending=False).iterrows():
    # SI ES POSITIVA Y TENEMOS EL DINERO PARA PRESTAR, PRESTAMOS, GUARDAMOS INDICE Y ACTUALIZAMOS TOTAL PRESTADO
    if ((monto_solicitado_total + row['Monto_Solicitado']) < 2000000) and (row['Prob_Pago']>0.80):
        monto_solicitado_total += row['Monto_Solicitado']
        monto_a_ganar+= row['Tasa_Interes']*row['Monto_Solicitado']
        approved_indices.append(index)

# ACTUALIZAMOS EL ESTADO A 1, DE DICHAS SOLICITUDES QUE APROBAMOS
df_solicitudes_test_cruzado.loc[approved_indices, 'Estado'] = 1
monto_solicitado_total,monto_a_ganar


(1999900, 539882.8899999999)

In [80]:
df_resultados = df_solicitudes_test_cruzado[['ID_Solicitud', 'Estado']].copy()
df_resultados.rename(columns={'Estado': 'Decision_Aprobacion'}, inplace=True)

# Exportar a CSV
df_resultados.to_csv('decisiones_aprobacion.csv', index=False)

print('Archivo decisiones_aprobacion.csv creado exitosamente.')
display(df_resultados.head())

Archivo decisiones_aprobacion.csv creado exitosamente.


,ID_Solicitud,Decision_Aprobacion
0,32417,0
1,32418,1
2,32419,0
3,32420,0
4,32421,0
